In [1]:
%cd /content
!rm -rf RiskAware-Complaints-Engine
!git clone https://github.com/Nusultan11/RiskAware-Complaints-Engine.git
%cd /content/RiskAware-Complaints-Engine
!pip install -e .


/content
Cloning into 'RiskAware-Complaints-Engine'...
remote: Enumerating objects: 288, done.
remote: Counting objects: 100% (288/288), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 288 (delta 120), reused 241 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (288/288), 171.36 KiB | 19.04 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/RiskAware-Complaints-Engine
Obtaining file:///content/RiskAware-Complaints-Engine
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for risk-aware-complaints-engine (pyproject.toml) ... done
  Created wheel for risk-aware-complaints-engine: filename=risk_aware_complaints_engine-0.1.0-0.editable-py3-none-any.whl size=2362 sha256=1a257e3e5bc12d214e416eb5a01aa981d43849404ea76a06ac2a2d48b10b5799
  Stored in directory: /tmp/pip-ephem-wh

In [19]:
import os
import sys
import json
import glob
import shutil
from pathlib import Path
import zipfile
from google.colab import files

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

from google.colab import drive

os.chdir("/content/RiskAware-Complaints-Engine")
sys.path.insert(0, os.path.abspath("src"))

from risk_aware.config import load_project_configs

In [3]:
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

cuda: True
device: Tesla T4


In [4]:
drive.mount("/content/drive")

ROOT = "/content/RiskAware-Complaints-Engine"
os.makedirs(f"{ROOT}/data/raw", exist_ok=True)

cands = glob.glob("/content/drive/MyDrive/**/cfpb_complaints.csv", recursive=True)
assert cands, "cfpb_complaints.csv не найден в Google Drive"
src = cands[0]
dst = f"{ROOT}/data/raw/cfpb_complaints.csv"
shutil.copy(src, dst)
print("copied:", src, "->", dst)


Mounted at /content/drive
copied: /content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv -> /content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv


In [5]:
!PYTHONPATH=src python -u src/risk_aware/data/prepare.py

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


In [6]:
TEXT_COL = "consumer_complaint_narrative"
LABEL_COL = "category"

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
    df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()
    df = df[(df[TEXT_COL] != "") & (df[LABEL_COL] != "")].copy()
    if name == "train":
        train_df = df.reset_index(drop=True)
    elif name == "val":
        val_df = df.reset_index(drop=True)
    else:
        test_df = df.reset_index(drop=True)

labels = sorted(train_df[LABEL_COL].unique().tolist())
label_to_id = {l: i for i, l in enumerate(labels)}
id_to_label = {i: l for l, i in label_to_id.items()}

val_unknown = set(val_df[LABEL_COL].unique()) - set(label_to_id.keys())
test_unknown = set(test_df[LABEL_COL].unique()) - set(label_to_id.keys())

print("rows:", len(train_df), len(val_df), len(test_df))
print("num_labels:", len(label_to_id))
print("val_unknown:", len(val_unknown), "test_unknown:", len(test_unknown))

rows: 42611 10639 13355
num_labels: 91
val_unknown: 0 test_unknown: 0


In [7]:
model_name = "distilbert-base-uncased"
max_length = 256
tokenizer = AutoTokenizer.from_pretrained(model_name)

def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    work = df[[TEXT_COL, LABEL_COL]].copy()
    work["labels"] = work[LABEL_COL].map(label_to_id)
    if work["labels"].isna().any():
        bad = work.loc[work["labels"].isna(), LABEL_COL].unique().tolist()
        raise ValueError(f"Unknown labels: {bad}")
    work["labels"] = work["labels"].astype(int)
    work = work[[TEXT_COL, "labels"]].reset_index(drop=True)
    return Dataset.from_pandas(work, preserve_index=False)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

train_ds = make_hf_dataset(train_df).map(tokenize_batch, batched=True)
val_ds = make_hf_dataset(val_df).map(tokenize_batch, batched=True)
test_ds = make_hf_dataset(test_df).map(tokenize_batch, batched=True)

print(train_ds)
print(train_ds[0].keys(), len(train_ds[0]["input_ids"]), train_ds[0]["labels"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/42611 [00:00<?, ? examples/s]

Map:   0%|          | 0/10639 [00:00<?, ? examples/s]

Map:   0%|          | 0/13355 [00:00<?, ? examples/s]

Dataset({
    features: ['consumer_complaint_narrative', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 42611
})
dict_keys(['consumer_complaint_narrative', 'labels', 'input_ids', 'attention_mask']) 256 40


In [8]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels_np, preds),
        "macro_f1": f1_score(labels_np, preds, average="macro"),
        "weighted_f1": f1_score(labels_np, preds, average="weighted"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_to_id),
    id2label=id_to_label,
    label2id=label_to_id,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args_common = dict(
    output_dir="artifacts/category_transformer/distilbert_baseline",
    overwrite_output_dir=True,
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    save_total_limit=2,
    report_to="none",
)

try:
    training_args = TrainingArguments(eval_strategy="epoch", **args_common)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **args_common)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("num_labels =", len(label_to_id))
print("model ready")
print("trainer ready")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3646/3591467129.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


num_labels = 91
model ready
trainer ready


In [9]:
train_output = trainer.train()
val_metrics = trainer.evaluate(eval_dataset=val_ds)
test_pred = trainer.predict(test_ds)

test_logits = test_pred.predictions
test_labels = test_pred.label_ids
test_preds = np.argmax(test_logits, axis=-1)

test_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_preds)),
    "macro_f1": float(f1_score(test_labels, test_preds, average="macro")),
    "weighted_f1": float(f1_score(test_labels, test_preds, average="weighted")),
}

print("VAL:", val_metrics)
print("TEST:", test_metrics)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.425000,1.705425,0.510010,0.130596,0.455738
2,1.570100,1.522302,0.560955,0.183563,0.525389
3,1.350200,1.483826,0.569602,0.190848,0.535192


VAL: {'eval_loss': 1.4838262796401978, 'eval_accuracy': 0.5696024062411881, 'eval_macro_f1': 0.19084841039747547, 'eval_weighted_f1': 0.5351922341714661, 'eval_runtime': 22.8201, 'eval_samples_per_second': 466.211, 'eval_steps_per_second': 14.592, 'epoch': 3.0}
TEST: {'accuracy': 0.5578435043055036, 'macro_f1': 0.19377041504582523, 'weighted_f1': 0.5230615263565211}


In [10]:
max_length = 384
print("max_length =", max_length)

max_length = 384


In [11]:
train_ds = make_hf_dataset(train_df).map(tokenize_batch, batched=True)
val_ds = make_hf_dataset(val_df).map(tokenize_batch, batched=True)
test_ds = make_hf_dataset(test_df).map(tokenize_batch, batched=True)

print(train_ds)
print(train_ds[0].keys())
print("input_len:", len(train_ds[0]["input_ids"]))


Map:   0%|          | 0/42611 [00:00<?, ? examples/s]

Map:   0%|          | 0/10639 [00:00<?, ? examples/s]

Map:   0%|          | 0/13355 [00:00<?, ? examples/s]

Dataset({
    features: ['consumer_complaint_narrative', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 42611
})
dict_keys(['consumer_complaint_narrative', 'labels', 'input_ids', 'attention_mask'])
input_len: 384


In [15]:
args_common = dict(
    output_dir="artifacts/category_transformer/distilbert_len384",
    overwrite_output_dir=True,
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    save_total_limit=2,
    report_to="none",
)

try:
    training_args = TrainingArguments(eval_strategy="epoch", **args_common)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **args_common)

In [16]:
trainer = Trainer(
    model=AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label_to_id),
        id2label=id_to_label,
        label2id=label_to_id,
    ),
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("trainer ready")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainer ready


/tmp/ipykernel_3646/3555187578.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
train_output = trainer.train()
val_metrics = trainer.evaluate(eval_dataset=val_ds)
test_pred = trainer.predict(test_ds)

test_logits = test_pred.predictions
test_labels = test_pred.label_ids
test_preds = np.argmax(test_logits, axis=-1)

test_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_preds)),
    "macro_f1": float(f1_score(test_labels, test_preds, average="macro")),
    "weighted_f1": float(f1_score(test_labels, test_preds, average="weighted")),
}

print("VAL:", val_metrics)
print("TEST:", test_metrics)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,2.426000,1.693040,0.516026,0.135121,0.464072
2,1.556200,1.504911,0.565091,0.189768,0.528591
3,1.339400,1.469815,0.575524,0.203712,0.540471


VAL: {'eval_loss': 1.4698153734207153, 'eval_accuracy': 0.5755240154149827, 'eval_macro_f1': 0.20371203228574977, 'eval_weighted_f1': 0.5404710599804201, 'eval_runtime': 36.3173, 'eval_samples_per_second': 292.946, 'eval_steps_per_second': 9.169, 'epoch': 3.0}
TEST: {'accuracy': 0.5612130288281543, 'macro_f1': 0.2001800468901545, 'weighted_f1': 0.5260938030059074}


In [18]:
out_metrics = Path("reports/metrics")
out_metrics.mkdir(parents=True, exist_ok=True)

out_model = Path("artifacts/category_transformer/distilbert_len384")
out_model.mkdir(parents=True, exist_ok=True)

(out_metrics / "category_transformer_len384_val.json").write_text(
    json.dumps(val_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(out_metrics / "category_transformer_len384_test.json").write_text(
    json.dumps(test_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

(out_model / "label_to_id.json").write_text(
    json.dumps(label_to_id, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(out_model / "id_to_label.json").write_text(
    json.dumps({str(k): v for k, v in id_to_label.items()}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

summary = {
    "model_name": "distilbert-base-uncased",
    "max_length": 384,
    "best_checkpoint_metric": "macro_f1",
    "best_epoch": 3,
    "val": val_metrics,
    "test": test_metrics,
}
(out_metrics / "category_transformer_len384_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("saved metrics and mappings")

saved metrics and mappings


In [23]:
root = Path("/content/RiskAware-Complaints-Engine")
bundle = root / "distilbert_len384_bundle.zip"

paths_to_pack = [
    "artifacts/category_transformer/distilbert_len384",
    "reports/metrics/category_transformer_len384_val.json",
    "reports/metrics/category_transformer_len384_test.json",
    "reports/metrics/category_transformer_len384_summary.json",
    "configs/category.yaml",
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in paths_to_pack:
        p = root / rel
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(root)))
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f.relative_to(root)))

print("created:", bundle)
files.download(str(bundle))

created: /content/RiskAware-Complaints-Engine/distilbert_len384_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
out_metrics = Path("reports/metrics")
out_metrics.mkdir(parents=True, exist_ok=True)

val_256 = {
    "eval_loss": 1.4838262796401978,
    "eval_accuracy": 0.5696024062411881,
    "eval_macro_f1": 0.19084841039747547,
    "eval_weighted_f1": 0.5351922341714661,
    "epoch": 3.0,
}
test_256 = {
    "accuracy": 0.5578435043055036,
    "macro_f1": 0.19377041504582523,
    "weighted_f1": 0.5230615263565211,
}

(out_metrics / "category_transformer_len256_val.json").write_text(
    json.dumps(val_256, indent=2, ensure_ascii=False), encoding="utf-8"
)
(out_metrics / "category_transformer_len256_test.json").write_text(
    json.dumps(test_256, indent=2, ensure_ascii=False), encoding="utf-8"
)

summary_256 = {
    "model_name": "distilbert-base-uncased",
    "max_length": 256,
    "best_checkpoint_metric": "macro_f1",
    "best_epoch": 3,
    "val": val_256,
    "test": test_256,
}
(out_metrics / "category_transformer_len256_summary.json").write_text(
    json.dumps(summary_256, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("saved len256 metrics")

saved len256 metrics


In [22]:
root = Path("/content/RiskAware-Complaints-Engine")
bundle = root / "distilbert_len256_bundle.zip"

paths_to_pack = [
    "reports/metrics/category_transformer_len256_val.json",
    "reports/metrics/category_transformer_len256_test.json",
    "reports/metrics/category_transformer_len256_summary.json",
    "configs/category.yaml",
]

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in paths_to_pack:
        p = root / rel
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(root)))

print("created:", bundle)
files.download(str(bundle))

created: /content/RiskAware-Complaints-Engine/distilbert_len256_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>